In [1]:
import pandas as pd

In [2]:
import pickle

In [3]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error

In [4]:
import mlflow

mlflow.set_tracking_uri('sqlite:////home/ubuntu/Mlops-course/03-experiment-tracking/mlflow.db')
mlflow.set_experiment('nyc_taxi_experiment')


<Experiment: artifact_location='/home/ubuntu/Mlops-course/02-ml-model/mlruns/1', creation_time=1783744047989, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783744047989, lifecycle_stage='active', name='nyc_taxi_experiment', tags={}, trace_location=None, workspace='default'>

In [7]:
df = pd.read_parquet("../data/green_tripdata_2021-01.parquet")

df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
df.duration = df.duration.apply(lambda td: td.total_seconds() / 60 )

df = df[(df.duration >= 1) & (df.duration <= 60)]

categorical = ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

df[categorical] = df[categorical].astype(str)

In [6]:
train_dicts = df[categorical + numerical].to_dict(orient='records')

dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

target = 'duration'
Y_train = df[target].values

lr = LinearRegression()
lr.fit(X_train, Y_train)

y_pred = lr.predict(X_train)

mean_squared_error(Y_train, y_pred)

96.80198150112638

In [8]:
sns.distplot(y_pred, label = 'prediction')
sns.distplot(Y_train, label = 'actual')

plt.legend()

NameError: name 'y_pred' is not defined

In [9]:
def read_dataframe(fileName): #"./notebook/data/green_tripdata_2021-01.parquet"
    df = pd.read_parquet(fileName)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60 )
    
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df

In [10]:
df_train = read_dataframe("../data/green_tripdata_2021-01.parquet")
df_val =read_dataframe("../data/green_tripdata_2021-02.parquet")

In [11]:
len(df_train), len(df_val)

(73908, 61921)

In [12]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [13]:
categorical = ['PU_DO'] # 'PULocationID', 'DOLocationID'
numerical = ['trip_distance']
dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)



In [14]:
target = 'duration'
Y_train = df_train[target].values
y_val = df_val[target].values

In [15]:
lr = LinearRegression()
lr.fit(X_train, Y_train)

y_pred = lr.predict(X_val)

mean_squared_error(y_val, y_pred)

60.197661613134706

In [16]:
with open('../models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [17]:
with mlflow.start_run():
    
    mlflow.set_tag("developer", "Alfredo")

    mlflow.log_param('train-data-path','../data/green_tripdata_2021-01.parquet')
    mlflow.log_param('valid-data-path','../data/green_tripdata_2021-02.parquet')

    alpha = 0.01    
    mlflow.log_param("alpha", alpha)
    lr = Lasso(alpha)
    lr.fit(X_train, Y_train)

    y_pred = lr.predict(X_val)
    rmse = mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    mlflow.log_artifact(local_path="../models/lin_reg.bin", artifact_path="models_pickle")

In [18]:
import xgboost as xgb

In [19]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

/home/ubuntu/anaconda3/lib/python3.13/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [20]:
train = xgb.DMatrix(X_train, label=Y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [21]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [26]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 20, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

/home/ubuntu/anaconda3/lib/python3.13/site-packages/xgboost/callback.py:385: UserWarning: [02:57:17] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:275: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:8.49714                           
[1]	validation-rmse:7.30410                           
[2]	validation-rmse:6.95105                           
[3]	validation-rmse:6.84282                           
[4]	validation-rmse:6.79886                           
[5]	validation-rmse:6.77692                           
[6]	validation-rmse:6.77171                           
[7]	validation-rmse:6.76741                           
[8]	validation-rmse:6.76310                           
[9]	validation-rmse:6.75712                           
[10]	validation-rmse:6.74989                          
[11]	validation-rmse:6.74294                          
[12]	validation-rmse:6.73945                          
[13]	validation-rmse:6.73463                          
[14]	validation-rmse:6.72604                          
[15]	validation-rmse:6.72303                          
[16]	validation-rmse:6.71914                          
[17]	validation-rmse:6.71673                          
[18]	valid

/home/ubuntu/anaconda3/lib/python3.13/site-packages/xgboost/callback.py:385: UserWarning: [02:58:38] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:275: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:9.22997                                                      
[1]	validation-rmse:7.78716                                                      
[2]	validation-rmse:7.13399                                                      
[3]	validation-rmse:6.84434                                                      
[4]	validation-rmse:6.70494                                                      
[5]	validation-rmse:6.64044                                                      
[6]	validation-rmse:6.60374                                                      
[7]	validation-rmse:6.58217                                                      
[8]	validation-rmse:6.56798                                                      
[9]	validation-rmse:6.55941                                                      
[10]	validation-rmse:6.55418                                                     
[11]	validation-rmse:6.55146                                                     
[12]	validation-

...

[0]	validation-rmse:11.69050                                                      
[1]	validation-rmse:11.21105                                                      
[2]	validation-rmse:10.77281                                                      
[3]	validation-rmse:10.37296                                                      
[4]	validation-rmse:10.00789                                                      
[5]	validation-rmse:9.67695                                                       
[6]	validation-rmse:9.37553                                                       
[7]	validation-rmse:9.10209                                                       
[8]	validation-rmse:8.85416                                                       
[9]	validation-rmse:8.63033                                                       
[10]	validation-rmse:8.42908                                                      
[11]	validation-rmse:8.24666                                                      
[12]

/home/ubuntu/anaconda3/lib/python3.13/site-packages/xgboost/callback.py:385: UserWarning: [04:09:47] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:275: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:10.58973                                                      
[1]	validation-rmse:9.40919                                                       
[2]	validation-rmse:8.56680                                                       
[3]	validation-rmse:7.97491                                                       
[4]	validation-rmse:7.56520                                                       
[5]	validation-rmse:7.28221                                                       
[6]	validation-rmse:7.08834                                                       
[7]	validation-rmse:6.95449                                                       
[8]	validation-rmse:6.86112                                                       
[9]	validation-rmse:6.79381                                                       
[10]	validation-rmse:6.74455                                                      
[11]	validation-rmse:6.71126                                                      
[12]

/home/ubuntu/anaconda3/lib/python3.13/site-packages/xgboost/callback.py:385: UserWarning: [04:11:32] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:275: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:10.19449                                                      
[1]	validation-rmse:8.86381                                                       
[2]	validation-rmse:8.01525                                                       
[3]	validation-rmse:7.48768                                                       
[4]	validation-rmse:7.16096                                                       
[5]	validation-rmse:6.96089                                                       
[6]	validation-rmse:6.83624                                                       
[7]	validation-rmse:6.75424                                                       
[8]	validation-rmse:6.70283                                                       
[9]	validation-rmse:6.66915                                                       
[10]	validation-rmse:6.64516                                                      
[11]	validation-rmse:6.62811                                                      
[12]

In [22]:
mlflow.xgboost.autolog(disable=True)

In [23]:
with mlflow.start_run():
    
    train = xgb.DMatrix(X_train, label=Y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.09585355369315604,
        'max_depth': 30,
        'min_child_weight': 1.060597050922164,
        'objective': 'reg:linear',
        'reg_alpha': 0.018060244040060163,
        'reg_lambda': 0.011658731377413597,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    rmse = mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("../models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    
    mlflow.log_artifact("../models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

/home/ubuntu/anaconda3/lib/python3.13/site-packages/xgboost/callback.py:385: UserWarning: [03:00:53] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:275: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:11.44174
[1]	validation-rmse:10.76627
[2]	validation-rmse:10.17716
[3]	validation-rmse:9.66439
[4]	validation-rmse:9.21865
[5]	validation-rmse:8.83455
[6]	validation-rmse:8.50406
[7]	validation-rmse:8.22010
[8]	validation-rmse:7.97540
[9]	validation-rmse:7.76692
[10]	validation-rmse:7.58908
[11]	validation-rmse:7.43668
[12]	validation-rmse:7.30705
[13]	validation-rmse:7.19678
[14]	validation-rmse:7.10367
[15]	validation-rmse:7.02340
[16]	validation-rmse:6.95494
[17]	validation-rmse:6.89584
[18]	validation-rmse:6.84617
[19]	validation-rmse:6.80219
[20]	validation-rmse:6.76453
[21]	validation-rmse:6.73295
[22]	validation-rmse:6.70496
[23]	validation-rmse:6.68006
[24]	validation-rmse:6.65942
[25]	validation-rmse:6.64052
[26]	validation-rmse:6.62474
[27]	validation-rmse:6.61037
[28]	validation-rmse:6.59772
[29]	validation-rmse:6.58649
[30]	validation-rmse:6.57661
[31]	validation-rmse:6.56756
[32]	validation-rmse:6.55900
[33]	validation-rmse:6.55050
[34]	validation-rmse:

2026/07/17 03:03:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [24]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import LinearSVR

mlflow.sklearn.autolog()

for model_class in (RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, LinearSVR):

    with mlflow.start_run():

        mlflow.log_param("train-data-path", "../data/green_tripdata_2021-01.csv")
        mlflow.log_param("valid-data-path", "../data/green_tripdata_2021-02.csv")
        mlflow.log_artifact("../models/preprocessor.b", artifact_path="preprocessor")

        mlmodel = model_class()
        mlmodel.fit(X_train, Y_train)

        y_pred = mlmodel.predict(X_val)
        rmse = mean_squared_error(y_val, y_pred )
        mlflow.log_metric("rmse", rmse)
        

: 